In [1]:
import sys
sys.path.append('/Users/kellyhui/koskella_lab_notebook')
from scipy.stats import linregress
from tools import *

In [2]:
dataframes = []
for i in range(1, 7):
    file_path = f'/Users/kellyhui/Downloads/RBG_data/4_24_26/KH_RBG_otr_042426_KF_P2_post_{i}.csv'
    df = pd.read_csv(f'{file_path}') 
    df = clean_and_transpose(df, 24, 122).dropna().transpose()
    df = df.reset_index()
    df.columns = ['Well', 'OD']
    dataframes.append(df)

#assign phage doses
def tidy_plate(df, plate_num):

    # Extract BioRep and column number
    df['BioRep'] = df['Well'].str[0]
    df['Col'] = df['Well'].str[1:].astype(int)

    # Map column number to dose
    def col_to_dose(col):
        if col in [1,2,3]:
            return 0
        elif col in [4,5,6]:
            return 1e4
        elif col in [7,8,9]:
            return 1e6
        elif col in [10,11,12]:
            return 1e8
        else:
            return np.nan

    df['Dose'] = df['Col'].apply(col_to_dose)
    df = df.sort_values(by=['BioRep', 'Dose', 'Col']).reset_index(drop=True)

    H_mean = df[df['BioRep'] == 'H']['OD'].mean()
    print(H_mean)
    df['OD_subtracted'] = df['OD'] - H_mean

    df = df[df['BioRep'] != 'H']

    df['Plate'] = plate_num

    return df[['Plate','BioRep','Dose','Col','OD', 'OD_subtracted']]

tbl_arr = [tidy_plate(dataframes[i], i+1) for i in range(6)]

0.1475749985833333
0.1338249983333333
0.12514166775
0.11749999900000002
0.14046666816666667
0.3579166629166666


In [3]:
#group plate, biol rep, and dose
bio_rep_mean = [tbl.groupby(['Plate','BioRep','Dose'], as_index=False)['OD'].mean() for tbl in tbl_arr]
bio_rep_mean[0].head(6)

,Plate,BioRep,Dose,OD
0,1,A,0.0,0.916233
1,1,A,10000.0,0.894233
2,1,A,1000000.0,0.848167
3,1,A,100000000.0,0.750033
4,1,B,0.0,0.827533
5,1,B,10000.0,0.888933


In [4]:
plate_results = []

for df in tbl_arr:
    df = df.copy()

    # log transform (handle 0)
    df["Dose_log"] = np.log10(df["Dose"] + 0.01)

    x = df["Dose_log"].values
    y = df["OD"].values

    if len(x) < 2:
        continue

    slope, intercept, r_value, p_value, _ = linregress(x, y)

    plate_results.append({
        "Plate": df["Plate"].iloc[0],
        "R2": r_value**2,
        "p_value": p_value,
        "slope": slope
    })

plate_table = pd.DataFrame(plate_results)
plate_table 

,Plate,R2,p_value,slope
0,1,0.095268,0.004282,-0.012431
1,2,0.049233,0.042510,-0.009097
2,3,0.000186,0.901906,-0.000676
3,4,0.051042,0.038790,-0.008996
4,5,0.000027,0.962696,0.000169
5,6,0.014646,0.272813,-0.002718


In [2]:
for i, df in enumerate(bio_rep_mean):
    plt.figure()

    df = df.copy()
    df['Dose_plot'] = df['Dose'].apply(lambda x: 0 if x == 0 else np.log10(x))

    x_all = df['Dose_plot'].values
    y_all = df['OD'].values

    slope, intercept, r_value, p_value, _ = linregress(x_all, y_all)
    r2 = r_value**2

    #plot each BioRep
    for bio in df['BioRep'].unique():
        bio_df = df[df['BioRep'] == bio]

        plt.plot(
            bio_df['Dose_plot'],
            bio_df['OD'],
            marker='o',
            alpha=0.6,
            label=f'BioRep {bio}'
        )


    #plot the regression line
    x_line = np.linspace(min(x_all), max(x_all), 100)
    y_line = slope * x_line + intercept
    plt.plot(x_line, y_line, linestyle='--', linewidth=2, color='black')

    # --- labels ---
    plt.xlabel('log10(PFU)')
    plt.ylabel('OD')
    plt.title(
        f'Population {i+1} (R²={r2:.2f}, p={p_value:.2g})'
    )

    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.ylim(0.1, 1.4)

    plt.show()

NameError: name 'bio_rep_mean' is not defined

In [10]:
bio_rep_mean = [tbl.groupby(['Plate','BioRep','Dose'], as_index=False)['OD'].mean() for tbl in tbl_arr]
combined_df = pd.concat(bio_rep_mean, ignore_index=True)

combined_df = combined_df[combined_df['Plate'] != 6]
variance_summary = (
    combined_df
    .groupby(['Plate', 'Dose'])
    .agg(
        mean_OD=('OD', 'mean'),
        variance_OD=('OD', 'var'),
        sd_OD=('OD', 'std')
    )
    .reset_index()
)

variance_summary['CV'] = (
    variance_summary['sd_OD'] /
    variance_summary['mean_OD']
)

variance_summary['CV'].mean()



np.float64(0.16226727178583727)